In [1]:
# pip install torch
# pip install transformers
# pip install ipywidgets
# pip install einops
# pip install triton

In [1]:
# pip install datasets accelerate scikit-learn

In [1]:
# import sys
# import os

# site_packages = None
# for p in sys.path:
#     if 'site-packages' in p.lower():
#         site_packages = p
#         break

# if site_packages:
#     triton_path = os.path.join(site_packages, 'triton')
#     os.makedirs(triton_path, exist_ok=True)
#     open(os.path.join(triton_path, '__init__.py'), 'a').close()

In [2]:
import torch
from transformers import AutoTokenizer, AutoModel

In [3]:
tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)

In [4]:
model = AutoModel.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)

C:\Users\34601\.cache\huggingface\modules\transformers_modules\zhihan1996\DNABERT_hyphen_2_hyphen_117M\7bce263b15377fc15361f52cfab88f8b586abda0\bert_layers.py:126: UserWarning: Unable to import Triton; defaulting MosaicBERT attention implementation to pytorch (this will reduce throughput when using this model).
  warnings.warn(
Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
dna = "ACGTAGCATCGGATCTATCTATCGACACTTGGTTATCGATCTACGAGCATCTCGTTAGC"

inputs = tokenizer(dna, return_tensors = 'pt')["input_ids"]
hidden_states = model(inputs)[0] # [1, sequence_length, 768]

# embedding with mean pooling
embedding_mean = torch.mean(hidden_states[0], dim = 0)
print(embedding_mean.shape) # expect to be 768

# embedding with max pooling
embedding_max = torch.max(hidden_states[0], dim = 0)[0]
print(embedding_max.shape) # expect to be 768

torch.Size([768])
torch.Size([768])


In [7]:
import pandas as pd
import numpy as np
import requests
import time
import mygene

from Bio import Entrez, SeqIO
from Bio.Seq import Seq

In [8]:
base = pd.read_csv("datosGene4PD/base_nueva_1332.csv", sep = ",", index_col = False)

In [9]:
base

,Secuencia,Etiqueta
0,ccagctccagtcacgccggaagcgcgggcggagcgcacgggtccgg...,Sano
1,ccagctccagtcacgccggaagcgcgggcggagcgcacgggtccgg...,Riesgo_PD
2,agaaggcggagcctacctctcatcaggaccagtctgactgcacctg...,Sano
3,agaaggcggagcctacctctcatcaggaccagtctgactgcacctg...,Riesgo_PD
4,cgcctcccgcccgctccgcagcgccagctcggcaggcgcggggcgt...,Sano
...,...,...
1327,atcaatgagatgcaaacatgaaagacaagaggaagaagaaggaccg...,Riesgo_PD
1328,tgtataagtggagtgtgctggggtgtgtaaagtagtatggaggcag...,Sano
1329,tgtataagtggagtgtgctggggtgtgtaaagtagtatggaggcag...,Riesgo_PD
1330,atctaaagggcattccgatggagcaggcaggctgcgccccgaaagg...,Sano


In [12]:
etiqueta_binaria = {"Sano": 0, "Riesgo_PD": 1}

In [13]:
base["Etiqueta"] = base["Etiqueta"].map(etiqueta_binaria)

In [14]:
base

,Secuencia,Etiqueta
0,ccagctccagtcacgccggaagcgcgggcggagcgcacgggtccgg...,0
1,ccagctccagtcacgccggaagcgcgggcggagcgcacgggtccgg...,1
2,agaaggcggagcctacctctcatcaggaccagtctgactgcacctg...,0
3,agaaggcggagcctacctctcatcaggaccagtctgactgcacctg...,1
4,cgcctcccgcccgctccgcagcgccagctcggcaggcgcggggcgt...,0
...,...,...
1327,atcaatgagatgcaaacatgaaagacaagaggaagaagaaggaccg...,1
1328,tgtataagtggagtgtgctggggtgtgtaaagtagtatggaggcag...,0
1329,tgtataagtggagtgtgctggggtgtgtaaagtagtatggaggcag...,1
1330,atctaaagggcattccgatggagcaggcaggctgcgccccgaaagg...,0
